# Pamoka 04 - Įrankių naudojimo dizaino šablonas

Šioje pamokoje sužinosite apie **Įrankių naudojimo** dizaino šabloną AI agentams, naudojant Microsoft Agent Framework (Python). Aptarsime:

- Funkcijų įrankių apibrėžimą naudojant `@tool` dekoratorių ir tipizuotus parametrus
- Įrankių schemų pateikimą, kad modelis žinotų, ką kiekvienas įrankis atlieka
- Įrankių vykdymo valdymą naudojant `approval_mode`
- **Struktūruoto išvesties** pateikimą per Pydantic modelius ir `response_format`

Scenarijus yra **kelionių užsakymo agentas**, kuris gali ieškoti kelionių tikslų, tikrinti prieinamumą ir gauti skrydžių informaciją.


## Diegimas


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Įrankių apibrėžimas naudojant @tool dekoratorių

`@tool` dekoratorius paverčia paprastą Python funkciją į įrankį, kurį gali iškviesti agentas.
Svarbiausi punktai:

- **docstring** tampa įrankio aprašymu, kurį mato modelis.
- **Tipų anotacijos** (įskaitant `Annotated` su aprašymais) apibrėžia įrankio schemą.
- `approval_mode` kontroliuoja, ar vartotojas turi patvirtinti kiekvieną kvietimą prieš jo įvykdymą.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Agentų kūrimas su keliomis įrankių priemonėmis

Perduokite visus tris įrankius klientui, kad modelis galėtų iškviesti bet kurį iš jų, kai tik reikia atsakyti į vartotojo klausimą.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Struktūruotas išvestis su įrankiais

Nustatant `response_format` kaip Pydantic modelį, agentas priverčiamas grąžinti gerai tipizuotą JSON objektą vietoj laisvos formos teksto. Tai naudinga, kai paskesnis kodas turi programiškai apdoroti rezultatą.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Įrankio patvirtinimo modeliai

`approval_mode` parametras `@tool` kontroliuoja, ar įrankio kvietimams reikia žmogaus patvirtinimo prieš vykdymą:

| Mode | Elgesys |
|---|---|
| `"never_require"` | Įrankis veikia automatiškai – nereikia vartotojo patvirtinimo. |
| `"always_require"` | Kiekvienas kvietimas turi būti patvirtintas vartotojo prieš vykdymą. |

Naudokite `"always_require"` įrankiams, turintiems šalutinių poveikių (pvz., bilietų užsakymas, kreditinės kortelės apmokestinimas), kad žmogus išliktų procese.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Santrauka

Šioje pamokoje jūs išmokote:

1. **Apibrėžti įrankius** naudojant `@tool` dekoratorių su tipizuotais parametrais ir dokumentacijos eilutėmis, kurios veikia kaip įrankio schema.
2. **Jungti kelis įrankius**, kad agentas galėtų juos iškviesti paeiliui atsakydamas į sudėtingus užklausimus.
3. **Grąžinti struktūruotą išvestį** perduodant Pydantic modelį kaip `response_format`.
4. **Valdyti įrankių patvirtinimą** naudojant `approval_mode`, kad jautriai veiklai išlaikyti žmogaus kontrolę.

Šie modeliai sudaro pagrindą patikimų, gamybai paruoštų agentų kūrimui, kurie gali saugiai sąveikauti su išorinėmis sistemomis.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės neperleidimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors stengiamės užtikrinti tikslumą, atkreipkite dėmesį, kad automatizuoti vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Kritinei informacijai rekomenduojamas profesionalus žmogaus vertimas. Mes neatsakome už jokius nesusipratimus ar neteisingus interpretavimus, kylančius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
